# Phase 2 Feedback TFIM-QRC v2.1 Compact-Summary Probe

The previous v2 probe used sequential dense/feedback QRC dynamics but exported the full observable trajectory, producing 580 readout features. That likely over-expanded noisy temporal dynamics.

This v2.1 probe keeps the sequential dense/feedback dynamics but compresses the trajectory into:

- final full observable vector;
- temporal mean/std of full observables;
- final/mean/std/min/max memory-Z summaries;
- final/mean/std/min/max readout feedback summaries.

Goal: test temporal memory without dumping the full 20-step trajectory into Ridge.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.feedback_tfim_reservoir import (
    FeedbackTFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_feedback_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_feedback_qrc_result,
)

## 1. Data and PCA-6 sequence windows

In [ ]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

display(pca6.explained_variance)
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})

## 2. Compact-summary feedback sweep

Same v2 dynamics, but `feature_collection="summary"`.

In [ ]:
summary_rows = []
summary_diag_rows = []
summary_results = {}

for feedback_gain in [0.0, 0.3, 0.7]:
    config = FeedbackTFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        temporal_steps=20,
        temporal_policy="even",
        input_qubits=(0, 1),
        memory_qubits=(2, 3, 4),
        readout_qubits=(5,),
        observable_mode="zxzz",
        feature_collection="summary",
        trotter_steps_per_time=1,
        evolution_time=0.25,
        input_scale=3.141592653589793 / 2,
        transverse_field=0.5,
        input_memory_coupling_scale=1.2,
        memory_coupling_scale=1.0,
        readout_coupling_scale=0.7,
        weak_background_coupling_scale=0.15,
        feedback_gain=float(feedback_gain),
        feedback_rotation="rz",
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        disorder_strength=0.10,
    )

    run_name = f"feedback_tfim_v21_summary_gain_{feedback_gain}"
    print(f"Running {run_name}")

    result = fit_feedback_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    summary_results[run_name] = result

    row = summarize_feedback_qrc_result(result)
    row["run_name"] = run_name
    summary_rows.append(row)

    _, y_train, _ = sequence_splits_6["train"]
    _, y_val, _ = sequence_splits_6["val"]
    _, y_test, _ = sequence_splits_6["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    summary_diag_rows.append(diag)

summary_table = pd.DataFrame(summary_rows)
summary_diagnostics = pd.concat(summary_diag_rows, ignore_index=True)

## 3. Metrics

In [ ]:
metric_cols = [
    "run_name",
    "feature_collection",
    "feedback_gain",
    "temporal_steps",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

summary_table[metric_cols].sort_values("test_rmse")

## 4. Diagnostics

In [ ]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

summary_diagnostics[diagnostic_cols]

## 5. Save outputs

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

summary_table.to_csv(out_dir / "phase2_feedback_tfim_qrc_v21_summary_probe.csv", index=False)
summary_diagnostics.to_csv(out_dir / "phase2_feedback_tfim_qrc_v21_summary_diagnostics.csv", index=False)

print("Saved v2.1 compact-summary outputs to", out_dir)

## 6. Interpretation rule

Compare against:

```text
v1 best static QRC:
RMSE  = 0.102618
QLIKE = -1.942716
MZ R² = 0.072134

v2 full-trajectory no-feedback:
RMSE  = 0.107123
QLIKE = -1.802335
MZ R² = 0.015167
```

Question: does compact temporal memory rescue v2 from the 580-feature overexpansion?